# 第 5 课：记忆的更新 — 冲突解决与增量策略

> 学完这节课，你会理解：当新信息和旧记忆冲突时怎么办、三种主流更新策略的优劣、你们系统的差分合并算法是怎么回事。

---

## 5.1 问题：新信息来了怎么办？

假设第一次聊天，你得知：

> 小明喜欢喝咖啡，每天早上必来一杯美式

三个月后，小明说：

> 我最近戒咖啡了，改喝茶了，龙井真不错

现在记忆系统里存着“喜欢咖啡”，新信息说“改喝茶了”。怎么办？

| 方案 | 操作 | 结果 |
|------|------|------|
| 直接覆盖 | 用新信息替换旧信息 | 丢失了“曾经喜欢咖啡”的历史 |
| 追加 | 新旧都保留 | “喜欢咖啡”和“改喝茶”同时存在，自相矛盾 |
| 智能合并 | 让 AI 判断怎么处理 | “以前喜欢咖啡，现在改喝茶了” |

这就是所有记忆系统都要面对的核心问题：**如何处理新旧信息的冲突**。

下面我们来看三种主流策略。

In [ ]:
# 环境准备
!pip install openai python-dotenv tiktoken -q

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

def call_llm(system_prompt: str, user_prompt: str, temperature: float = 0.3) -> str:
    """封装 LLM 调用，后续代码复用"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
    )
    return response.choices[0].message.content

print("环境就绪")

## 5.2 策略一：全量重写

最简单粗暴的方法：**每次有新聊天记录，就把所有历史记录全部重新扔给 AI，生成一份全新的摘要**。

```
第 1 天的聊天记录 ─┐
第 2 天的聊天记录 ─┤
...                ├──→ LLM ──→ 全新的摘要
第 N 天的聊天记录 ─┤
今天的聊天记录   ─┘
```

类比前端：就像每次 state 变化都重新渲染整个页面，而不是用 Virtual DOM diff。

In [ ]:
def full_rewrite(all_conversations: list) -> str:
    """策略一：全量重写 — 把所有聊天记录扔给 AI 重新生成摘要"""
    system_prompt = """你是一个记忆管理助手。请根据以下所有聊天记录，生成一份关于这个联系人的简洁摘要。
要求：
1. 提取关键信息：喜好、习惯、重要事件、关系变化
2. 如果信息有变化（比如以前喜欢 A，后来改喜欢 B），以最新的为准，但可以注明变化
3. 用简洁的条目格式输出"""

    all_text = "\n---\n".join(all_conversations)
    user_prompt = f"以下是与联系人【小明】的所有聊天记录：\n\n{all_text}"

    return call_llm(system_prompt, user_prompt)


# 模拟：随着时间推移，聊天记录越来越多
conversations = [
    "【1月】小明：最近迷上了咖啡，天天去瑞幸。我：什么口味？小明：美式，不加糖。",
    "【3月】小明：周末去打羽毛球吗？我：好啊，老地方？小明：嗯，我新买了个尤尼克斯的拍子。",
    "【5月】小明：我最近在学吉他。我：什么歌？小明：民谣，想学朴树的那些。",
    "【7月】小明：咖啡戒了，最近改喝茶了。我：为啥？小明：胃不好，喝龙井感觉不错。",
    "【9月】小明：吉他放弃了，太难了。最近在学摄影。我：拍什么？小明：街拍为主。",
]

# 第一次：只有 1 月的记录
print("=== 只有 1 月记录时的摘要 ===")
summary_v1 = full_rewrite(conversations[:1])
print(summary_v1)

print("\n")

# 后来：有了全部记录，重新生成
print("=== 有全部记录后重新生成的摘要 ===")
summary_v2 = full_rewrite(conversations)
print(summary_v2)

In [ ]:
import tiktoken

enc = tiktoken.encoding_for_model("gpt-4o-mini")

# 算算全量重写的 token 消耗增长
print("聊天记录累积 vs Token 消耗：\n")
for i in range(1, len(conversations) + 1):
    text = "\n---\n".join(conversations[:i])
    tokens = len(enc.encode(text))
    print(f"  累积 {i} 段记录 → {tokens} tokens → 约 ${tokens * 0.15 / 1_000_000 * 2:.4f}/次")

print("\n随着聊天记录增长，每次全量重写的成本线性增加。")
print("如果一个联系人有 1000 段对话，每次更新都要处理全部 —— 太贵了。")

### 全量重写的优缺点

| 优点 | 缺点 |
|------|------|
| 实现简单，不用管什么合并逻辑 | Token 消耗随聊天量线性增长 |
| 每次都是全局最优的摘要 | 大量聊天记录时又慢又贵 |
| 天然解决冲突（AI 看到全局） | 细节可能在压缩中丢失 |

适合：**聊天量小、更新频率低**的场景。

## 5.3 策略二：增量更新

**只处理新增的聊天记录，和旧摘要合并**，生成新的摘要。

```
旧摘要   ─┐
           ├──→ LLM ──→ 新摘要
新增记录 ─┘
```

类比前端：这就像 React 的 Virtual DOM diff —— 不重新渲染整个页面，只更新变化的部分。

这也是你们系统中 `memory_layer_executor` 采用的策略。

In [ ]:
def incremental_update(old_summary: str, new_conversations: list) -> str:
    """策略二：增量更新 — 旧摘要 + 新记录 → 新摘要"""
    system_prompt = """你是一个记忆管理助手。你会收到：
1. 一份已有的联系人摘要
2. 一些新的聊天记录

请根据新的聊天记录更新摘要。规则：
- 保留旧摘要中仍然有效的信息
- 如果新信息和旧信息冲突，以新信息为准（但可以注明变化）
- 新增的信息添加到摘要中
- 保持简洁的条目格式"""

    new_text = "\n".join(new_conversations)
    user_prompt = f"""【已有摘要】
{old_summary}

【新增聊天记录】
{new_text}"""

    return call_llm(system_prompt, user_prompt)


# 模拟增量更新过程
print("=== 第 1 次：处理 1 月记录 ===")
summary = incremental_update("（暂无信息）", [conversations[0]])
print(summary)

print("\n=== 第 2 次：处理 3 月记录（增量）===")
summary = incremental_update(summary, [conversations[1]])
print(summary)

print("\n=== 第 3 次：处理 5 月记录（增量）===")
summary = incremental_update(summary, [conversations[2]])
print(summary)

print("\n=== 第 4 次：处理 7 月记录（冲突：咖啡→茶）===")
summary = incremental_update(summary, [conversations[3]])
print(summary)

print("\n=== 第 5 次：处理 9 月记录（冲突：吉他→摄影）===")
summary = incremental_update(summary, [conversations[4]])
print(summary)

In [ ]:
# 对比增量更新的 token 消耗
print("增量更新 vs 全量重写的 token 消耗对比：\n")

# 假设每次摘要大约 200 字
summary_tokens = len(enc.encode("摘要大约两百字左右的样本文本用于估算" * 5))

for i in range(1, len(conversations) + 1):
    # 全量重写：所有记录
    full_text = "\n---\n".join(conversations[:i])
    full_tokens = len(enc.encode(full_text))

    # 增量更新：旧摘要 + 新记录
    new_text = conversations[i - 1]
    incr_tokens = summary_tokens + len(enc.encode(new_text))

    print(f"  第 {i} 次更新: 全量={full_tokens:>4} tokens, 增量={incr_tokens:>4} tokens, 节省={full_tokens - incr_tokens:>4}")

print("\n聊天记录越多，增量更新的优势越明显。")

### 你们系统中的增量更新参数

在 `memory_layer_executor` 中，增量更新不是每次聊天就触发的，有两个控制参数：

| 参数 | 含义 | 为什么需要 |
|------|------|----------|
| `cooldown`（冷却期） | 两次更新之间的最小间隔时间 | 避免频繁调 API 浪费钱，比如用户连续发 10 条消息，不需要更新 10 次 |
| `min_fact_count`（最小事实数） | 累积多少条新事实才触发更新 | 只有一两条新信息不值得更新，攒够了再一起处理 |

类比前端：
- `cooldown` 就像 **debounce** —— 用户疯狂输入时，等停下来再处理
- `min_fact_count` 就像 **批量更新** —— 攒够一批 state 变化再一次性 render

In [ ]:
import time

class IncrementalMemoryUpdater:
    """模拟你们系统的增量更新逻辑（含 cooldown 和 min_fact_count）"""

    def __init__(self, cooldown_seconds: float = 5.0, min_fact_count: int = 3):
        self.cooldown_seconds = cooldown_seconds
        self.min_fact_count = min_fact_count
        self.summary = "（暂无信息）"
        self.pending_facts: list = []  # 等待处理的新事实
        self.last_update_time = 0.0
        self.update_count = 0

    def add_fact(self, fact: str) -> str:
        """新事实到来，判断是否触发更新"""
        self.pending_facts.append(fact)

        # 检查冷却期
        time_since_last = time.time() - self.last_update_time
        if time_since_last < self.cooldown_seconds:
            return f"  冷却中（还需等 {self.cooldown_seconds - time_since_last:.1f}s），暂存事实，当前待处理: {len(self.pending_facts)} 条"

        # 检查最小事实数
        if len(self.pending_facts) < self.min_fact_count:
            return f"  事实不足（需要 {self.min_fact_count} 条，当前 {len(self.pending_facts)} 条），继续等待"

        # 触发更新
        return self._do_update()

    def _do_update(self) -> str:
        """执行增量更新"""
        self.update_count += 1
        result = f"  >>> 触发第 {self.update_count} 次更新！处理 {len(self.pending_facts)} 条新事实"
        self.pending_facts = []
        self.last_update_time = time.time()
        return result

    def force_update(self) -> str:
        """强制更新（不管冷却期和最小事实数）"""
        if not self.pending_facts:
            return "  没有待处理的事实"
        return self._do_update()


# 模拟一连串事实到来
updater = IncrementalMemoryUpdater(cooldown_seconds=2.0, min_fact_count=3)

facts = [
    "小明喜欢咖啡",
    "小明打羽毛球",
    "小明在学吉他",       # 第 3 条，触发更新
    "小明改喝茶了",       # 冷却期内
    "小明放弃吉他了",     # 冷却期内
    "小明开始学摄影",     # 冷却期过了 + 够 3 条，触发更新
]

print("模拟事实逐条到来的处理过程：\n")
for i, fact in enumerate(facts):
    print(f"[{i+1}] 新事实: {fact}")
    result = updater.add_fact(fact)
    print(result)
    if i == 2:
        time.sleep(2.5)  # 模拟时间流逝，让冷却期过去

# 最后强制处理剩余的
print("\n[结束] 强制处理剩余事实")
print(updater.force_update())

## 5.4 策略三：Mem0 的 ADD/UPDATE/DELETE/NOOP

[Mem0](https://github.com/mem0ai/mem0) 是一个开源记忆系统，它的更新策略更精细：

**对每一条新记忆，让 LLM 决定执行什么操作**：

| 操作 | 含义 | 例子 |
|------|------|------|
| **ADD** | 新增一条记忆 | 新信息：“小明养了一只猫” |
| **UPDATE** | 更新已有记忆 | 旧：“喜欢咖啡” → 新：“改喝茶了” |
| **DELETE** | 删除已有记忆 | “不再打羽毛球了”（删除旧的运动信息） |
| **NOOP** | 不做任何操作 | 重复信息或无关紧要的内容 |

类比前端：这就像 Redux 的 action —— 每个变更都是一个明确的操作，有类型和 payload。

In [ ]:
def mem0_decide(existing_memories: list, new_memory: str) -> dict:
    """模拟 Mem0 的决策过程：让 LLM 决定对新记忆执行什么操作"""
    system_prompt = """你是一个记忆管理 AI。你会收到：
1. 一组已有的记忆条目
2. 一条新的信息

请决定对新信息执行什么操作。返回 JSON 格式：

如果是全新信息（已有记忆中没有相关内容）：
{"action": "ADD", "memory": "要添加的内容", "reason": "原因"}

如果需要更新某条已有记忆：
{"action": "UPDATE", "target_index": 被更新的记忆序号, "new_memory": "更新后的内容", "reason": "原因"}

如果需要删除某条已有记忆：
{"action": "DELETE", "target_index": 被删除的记忆序号, "reason": "原因"}

如果不需要做任何操作（重复或无意义信息）：
{"action": "NOOP", "reason": "原因"}

只返回 JSON，不要其他内容。"""

    memories_text = "\n".join(f"  [{i}] {m}" for i, m in enumerate(existing_memories))
    if not memories_text:
        memories_text = "  （空）"

    user_prompt = f"""已有记忆：
{memories_text}

新信息：{new_memory}"""

    result = call_llm(system_prompt, user_prompt, temperature=0)

    # 尝试解析 JSON
    try:
        cleaned = result.strip()
        if cleaned.startswith("```"):
            cleaned = cleaned.split("\n", 1)[1].rsplit("```", 1)[0]
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return {"action": "ERROR", "raw": result}


# 模拟 Mem0 的决策过程
memories: list = []

new_facts = [
    "小明喜欢喝美式咖啡",
    "小明每周打羽毛球",
    "小明喜欢喝美式咖啡",        # 重复 → NOOP
    "小明最近戒了咖啡，改喝茶",   # 冲突 → UPDATE
    "小明不打羽毛球了",           # 删除 → DELETE
    "小明开始学摄影",             # 新增 → ADD
]

print("=== Mem0 风格的记忆更新决策 ===\n")

for fact in new_facts:
    print(f'新信息: "{fact}"')
    print(f"当前记忆: {memories if memories else '（空）'}")

    decision = mem0_decide(memories, fact)
    print(f"决策: {json.dumps(decision, ensure_ascii=False, indent=2)}")

    # 执行决策
    action = decision.get("action")
    if action == "ADD":
        memories.append(decision["memory"])
    elif action == "UPDATE":
        idx = decision["target_index"]
        if 0 <= idx < len(memories):
            memories[idx] = decision["new_memory"]
    elif action == "DELETE":
        idx = decision["target_index"]
        if 0 <= idx < len(memories):
            memories.pop(idx)

    print(f"更新后记忆: {memories}")
    print("-" * 60)

In [ ]:
# 不调 API 的本地版本：手动模拟 Mem0 决策过程
# 方便理解逻辑，不花钱

def mem0_decide_local(existing: list, new_info: str) -> dict:
    """本地模拟 Mem0 决策（基于简单的关键词匹配）"""

    # 检查是否重复
    for mem in existing:
        if new_info in mem or mem in new_info:
            return {"action": "NOOP", "reason": "重复信息"}

    # 检查是否是否定/更新类信息
    negative_keywords = ["不再", "戒了", "放弃", "不打", "不喜欢"]
    update_keywords = ["改", "换成", "现在喜欢"]

    for i, mem in enumerate(existing):
        # 简单判断：新信息是否在否定旧记忆
        if any(kw in new_info for kw in negative_keywords):
            # 检查话题是否相关（有共同的名词）
            common_chars = set(mem) & set(new_info) - set('的了在是不我他她')
            if len(common_chars) > 2:
                return {"action": "DELETE", "target_index": i, "reason": f"新信息否定了旧记忆 [{mem}]"}

        if any(kw in new_info for kw in update_keywords):
            common_chars = set(mem) & set(new_info) - set('的了在是不我他她')
            if len(common_chars) > 2:
                return {"action": "UPDATE", "target_index": i, "new_memory": new_info, "reason": f"更新旧记忆 [{mem}]"}

    return {"action": "ADD", "memory": new_info, "reason": "全新信息"}


# 快速演示
demo_memories = ["小明喜欢咖啡", "小明每周打羽毛球"]
demo_new = ["小明改喝茶了", "小明喜欢咖啡", "小明开始学摄影", "小明不打羽毛球了"]

print("=== 本地模拟（不调 API）===\n")
for info in demo_new:
    result = mem0_decide_local(demo_memories, info)
    print(f"新信息: {info}")
    print(f"决策: {result}")

    # 执行决策以更新 demo_memories
    if result["action"] == "ADD":
        demo_memories.append(result["memory"])
    elif result["action"] == "UPDATE":
        demo_memories[result["target_index"]] = result["new_memory"]
    elif result["action"] == "DELETE":
        demo_memories.pop(result["target_index"])

    print(f"当前记忆: {demo_memories}\n")

## 5.5 你们系统的差分合并算法

你们系统的 `profile_auto` 合并逻辑比前面两种策略更精巧。它要解决一个额外的问题：

**用户可能会手动编辑记忆**。

想象这个场景：

1. AI 自动生成了小明的 profile：`{喜欢咖啡, 打羽毛球, 学吉他}`
2. 用户觉得“学吉他”不重要，**手动删了**
3. 用户觉得“喜欢川菜”很重要，**手动加了**
4. 现在 AI 又处理了新的聊天记录，想要更新 profile

问题来了：AI 的更新不能把用户手动删的加回去，也不能把用户手动加的弄丢。

### 三个关键变量

| 变量 | 含义 | 例子 |
|------|------|------|
| `old_auto` | 上一次 AI 自动生成的 profile | `{咖啡, 羽毛球, 吉他}` |
| `current_profile` | 用户当前看到的 profile（可能被手动编辑过） | `{咖啡, 羽毛球, 川菜}` |
| `patch` | AI 这次要做的变更（增/删） | add=`{摄影, 茶}`, remove=`{咖啡, 吉他}` |

### 合并步骤

```
第一步：检测用户的手动编辑
  manual_add    = current_profile - old_auto  → 用户手动加的
  manual_remove = old_auto - current_profile  → 用户手动删的

第二步：应用 AI 的 patch
  new_auto = (old_auto ∪ patch_add) - patch_remove

第三步：尊重用户的手动编辑
  final = (new_auto - manual_remove) ∪ manual_add
```

核心思想：**AI 的自动更新和用户的手动编辑互不干扰**。

In [ ]:
def diff_merge(
    old_auto: set,
    current_profile: set,
    patch_add: set,
    patch_remove: set,
) -> dict:
    """你们系统的差分合并算法 — 用 Python set 实现"""

    # 第一步：检测用户的手动编辑
    manual_add = current_profile - old_auto      # 用户手动加的
    manual_remove = old_auto - current_profile    # 用户手动删的

    # 第二步：应用 AI 的 patch
    new_auto = (old_auto | patch_add) - patch_remove

    # 第三步：尊重用户的手动编辑
    final = (new_auto - manual_remove) | manual_add

    return {
        "manual_add": manual_add,
        "manual_remove": manual_remove,
        "new_auto": new_auto,
        "final": final,
    }


# 演示场景
old_auto = {"喜欢咖啡", "打羽毛球", "学吉他"}
current_profile = {"喜欢咖啡", "打羽毛球", "喜欢川菜"}  # 删了"学吉他"，加了"喜欢川菜"
patch_add = {"学摄影", "喝茶"}       # AI 新发现的
patch_remove = {"喜欢咖啡", "学吉他"}  # AI 认为过时的

print("输入：")
print(f"  old_auto（上次 AI 生成的）     = {old_auto}")
print(f"  current_profile（用户当前看到的）= {current_profile}")
print(f"  patch_add（AI 要新增的）        = {patch_add}")
print(f"  patch_remove（AI 要删除的）     = {patch_remove}")

result = diff_merge(old_auto, current_profile, patch_add, patch_remove)

print("\n计算过程：")
print(f"  manual_add = current - old_auto = {result['manual_add']}   ← 用户手动加的")
print(f"  manual_remove = old_auto - current = {result['manual_remove']}   ← 用户手动删的")
print(f"  new_auto = (old_auto ∪ patch_add) - patch_remove = {result['new_auto']}")
print(f"  final = (new_auto - manual_remove) ∪ manual_add = {result['final']}")

print("\n最终结果：")
print(f"  {result['final']}")
print("\n验证：")
print(f"  '喜欢川菜' 保留了吗？{'喜欢川菜' in result['final']}  ← 用户手动加的，必须保留")
print(f"  '学吉他' 回来了吗？ {'学吉他' in result['final']}  ← 用户手动删的，不能回来")
print(f"  '喜欢咖啡' 还在吗？ {'喜欢咖啡' in result['final']}  ← AI 认为过时了，删掉")
print(f"  '学摄影' 加进去了吗？{'学摄影' in result['final']}  ← AI 新发现的，加入")
print(f"  '喝茶' 加进去了吗？  {'喝茶' in result['final']}  ← AI 新发现的，加入")

In [ ]:
# 多轮差分合并演示

def simulate_rounds():
    """模拟多轮更新，看差分合并怎么工作"""

    # 初始状态
    auto = set()       # AI 自动生成的部分
    profile = set()    # 用户实际看到的

    rounds = [
        {
            "name": "第 1 轮：AI 首次分析聊天记录",
            "user_edit": False,
            "patch_add": {"喜欢咖啡", "打羽毛球", "学吉他", "住在北京"},
            "patch_remove": set(),
        },
        {
            "name": "第 2 轮：用户手动编辑（AI 未更新）",
            "user_edit": True,
            "patch_add": set(),
            "patch_remove": set(),
        },
        {
            "name": "第 3 轮：AI 处理新聊天记录",
            "user_edit": False,
            "patch_add": {"学摄影", "喝茶"},
            "patch_remove": {"喜欢咖啡", "学吉他"},
        },
        {
            "name": "第 4 轮：AI 又处理新聊天记录",
            "user_edit": False,
            "patch_add": {"养了一只猫"},
            "patch_remove": {"打羽毛球"},
        },
    ]

    for r in rounds:
        print(f"\n{'='*50}")
        print(f"{r['name']}")
        print(f"{'='*50}")

        # 用户手动编辑
        if r["user_edit"]:
            profile = (profile - {"学吉他"}) | {"喜欢川菜"}
            print(f"用户编辑后 profile = {profile}")
            continue

        if not r["patch_add"] and not r["patch_remove"]:
            continue

        print(f"  AI patch: +{r['patch_add']} -{r['patch_remove']}")

        result = diff_merge(auto, profile, r["patch_add"], r["patch_remove"])

        print(f"  检测到用户手动新增: {result['manual_add']}")
        print(f"  检测到用户手动删除: {result['manual_remove']}")

        # 更新状态
        auto = result["new_auto"]
        profile = result["final"]

        print(f"  new_auto = {auto}")
        print(f"  最终 profile = {profile}")


simulate_rounds()

### 为什么要这么设计？

想象你是用户：

1. 你费劲手动删掉了某条信息 → **绝对不能让 AI 自动加回来**
2. 你费劲手动加了某条信息 → **绝对不能让 AI 自动删掉**
3. AI 发现了新信息 → 加进来，没问题
4. AI 认为旧信息过时了 → 删掉，但前提是这不是用户手动加的

这就是“**用户意图优先**”的设计原则。

类比前端：就像 `git merge` —— 两个人同时编辑代码，git 要搞清楚哪些是你改的、哪些是别人改的，然后合并。

## 5.6 对比三种策略

| | 全量重写 | 增量更新 | Mem0 ADD/UPDATE/DELETE |
|---|---|---|---|
| **复杂度** | 低 | 中 | 高 |
| **Token 成本** | 高（随数据量线性增长） | 低（只处理增量） | 中（每条新信息一次 LLM 调用） |
| **冲突处理** | 好（AI 看到全局） | 中（依赖 prompt 指导） | 好（逐条精确决策） |
| **用户编辑兼容** | 差（覆盖用户编辑） | 可以做（需要差分合并） | 好（可以标记保护条目） |
| **适用场景** | 数据量小、更新少 | 数据量大、需要省钱 | 需要精确控制每条记忆 |
| **代表系统** | 简单的 chatbot | 你们的 memory_layer | Mem0 |

In [ ]:
# 用代码直观对比三种策略的成本

def estimate_costs(total_facts: int, new_facts_per_batch: int = 5) -> dict:
    """
    估算三种策略在不同数据量下的 API 调用成本
    假设：每条事实 ~50 tokens，摘要 ~200 tokens
    """
    tokens_per_fact = 50
    summary_tokens = 200

    num_batches = total_facts // new_facts_per_batch

    # 全量重写：每次都处理所有事实
    full_rewrite_tokens = sum(
        (i + 1) * new_facts_per_batch * tokens_per_fact
        for i in range(num_batches)
    )

    # 增量更新：每次处理摘要 + 新事实
    incremental_tokens = num_batches * (summary_tokens + new_facts_per_batch * tokens_per_fact)

    # Mem0：每条新事实单独处理（对比已有记忆）
    mem0_tokens = sum(
        summary_tokens + tokens_per_fact
        for _ in range(total_facts)
    )

    return {
        "全量重写": full_rewrite_tokens,
        "增量更新": incremental_tokens,
        "Mem0": mem0_tokens,
    }


print("三种策略的 Token 消耗对比\n")
print(f"{'事实总数':>8} | {'全量重写':>10} | {'增量更新':>10} | {'Mem0':>10}")
print("-" * 50)

for n in [10, 50, 100, 500, 1000]:
    costs = estimate_costs(n)
    print(f"{n:>8} | {costs['全量重写']:>10,} | {costs['增量更新']:>10,} | {costs['Mem0']:>10,}")

print("\n单位：tokens")
print("\n结论：数据量越大，增量更新的优势越明显。")
print("Mem0 的总量比增量更新多，但胜在每条记忆都精确可控。")

## 5.7 本课小结

| 概念 | 一句话解释 | 你们系统中的对应 |
|------|----------|---------------|
| **全量重写** | 每次把全部记录重新处理生成摘要 | 适合初始化，不适合常态更新 |
| **增量更新** | 只处理新增记录，和旧摘要合并 | `memory_layer_executor` 的核心策略 |
| **cooldown** | 两次更新之间的最小间隔 | 避免频繁调 API（类似 debounce） |
| **min_fact_count** | 触发更新的最小新事实数 | 攒够一批再处理（类似批量更新） |
| **Mem0 决策** | 对每条新记忆决定 ADD/UPDATE/DELETE/NOOP | 参考思路，精确控制每条记忆 |
| **差分合并** | 用集合运算合并 AI 更新和用户编辑 | `profile_auto` 的合并逻辑 |
| **用户意图优先** | 用户手动编辑不能被 AI 覆盖 | 差分合并的核心原则 |

### 关键认识

记忆更新不是简单的“新替旧”。一个好的记忆系统需要：

1. **省钱** —— 不要每次都重新处理所有数据（增量更新）
2. **准确** —— 正确处理冲突，新信息覆盖旧的矛盾信息
3. **尊重用户** —— 用户的手动编辑永远优先于 AI 的自动更新

### 下节预告

下一课我们学习记忆的检索 —— 当记忆越来越多，怎么快速找到和当前对话相关的那些？这涉及到向量搜索和语义匹配。